[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/093.%20The%20Inventory-Routing%20Problem%20%28IRP%29/P93-Tier-10_executed.ipynb)

---


# 93. The Inventory-Routing Problem (IRP)
## Tier 10: Proactive Demand and Market Shaping

### Purpose and scope

This notebook illustrates how demand-shaping strategies, partnership development, and market-influence mechanisms can be represented in a multi-period supply-chain simulation. The program applies these components in a fixed sequence, assigns customers to vehicles with a demand-sorted allocation heuristic, and reports financial and strategic indicators.

The routing step is a simple assignment heuristic, not an exact IRP optimizer. In particular, this model does not choose delivery quantities, update depot or customer inventories, enforce vehicle capacities, or optimize travel sequences. Consequently, its financial and strategic indicators describe this illustrative simulation; they are not evidence of a feasible or optimal inventory-routing plan.

### Inputs and outputs

Inputs are one depot, a generated set of customers, three vehicles, a five-period horizon, and the strategy and mechanism parameters defined in the code. The simulation produces period-by-period shaped demand, assigned customer lists, estimated revenue and costs, partnership states, and market-position indicators. Random demand effects and partnership formation mean that results depend on the random-number generators; both are seeded for reproducibility within a given software environment.

### Modeling assumptions

- Customer demand starts from each customer's fixed `demand_mean`; the generated `demand_std` is recorded but is not used to sample baseline demand.
- Demand-shaping effects multiply. Each strategy's effect is adjusted by its effectiveness and bounded individually, while the product of multiple strategies is not given an additional overall bound.
- Customer locations are measured in consistent distance units. Transportation cost uses the straight-line distance from the depot to each assigned customer, with no return-leg or inter-customer distance.
- Revenue is estimated at a base price of $10 per demand unit, with a partnership price multiplier. Strategy and partnership costs are simplified accounting assumptions.
- Market position is a bounded state updated each period from the mechanisms' influence scores. The competitor data in the landscape analysis are illustrative reference values, not observations from the simulation.

### Simulation logic and mathematical representation

A period applies the following stages in order: shape demand, develop and age partnerships, calculate mechanism influence and update market position, assign customers to vehicles, estimate financial performance, and calculate strategic value. The sequence is part of the model: for example, partnerships formed in a period can affect that period's mechanism scores and partner pricing.

For customer $i$ in period $t$, let $d_i$ be the baseline mean demand and let $E_{ist}$ be the demand multiplier from strategy $s$. The shaped demand is

$$d_{it}^{\mathrm{shaped}} = d_i \prod_{s \in S_i} E_{ist},$$

where $S_i$ is the set of strategies targeting customer $i$. In the code, `shape_demand` performs the product and `_apply_demand_shaping_strategy` calculates each multiplier. For pricing, the unadjusted multiplier is $1 + \epsilon_i p_{ist}$, where $\epsilon_i$ is the configured price elasticity and $p_{ist}$ is a randomly drawn price change. Other strategy types use their configured nudging, segmentation, or loyalty rules.

The route-assignment heuristic sorts customers by shaped demand in descending order and assigns consecutive portions of that list to vehicles. It does not optimize a mathematical routing objective. The estimated period cost is

$$C_t = C_t^{\mathrm{transport}} + C_t^{\mathrm{strategy}} + C_t^{\mathrm{partnership}},$$

where transportation cost includes each used vehicle's fixed cost and each assigned customer's depot-distance cost; strategy cost is the sum of strategy cost factors multiplied by period revenue; and partnership cost is a fixed charge per active partnership. Profit is $\Pi_t = R_t - C_t$, and profit margin is $\Pi_t/R_t$ when revenue is positive.

Strategic value combines four weighted indicators:

$$V_t = 0.4M_t + 0.3I_t + 0.2P_t + 0.1A_t,$$

where $M_t$ is profit margin, $I_t$ is mean mechanism influence, $P_t$ is the partnership contribution defined in the code, and $A_t$ is competitive advantage. The implementation is `_calculate_strategic_value`. This score is a constructed indicator, not a monetary quantity or a solver objective.

The simulation processes $T$ periods and, in each period, loops over customers, strategies, mechanisms, and assigned customers. Its basic computation therefore grows approximately in proportion to $T$ times the number of customers times the number of strategies, plus the corresponding mechanism and vehicle calculations. The random draws make the process stochastic; seeding makes a run repeatable under the same random-number-generator behavior.

### What to examine

Interpret period-by-period financial results alongside the market and partnership indicators. A rise in market share or strategic value does not by itself establish improved profitability, feasible vehicle loading, or better IRP performance. The comparison section uses explicitly labeled illustrative reference values and should not be read as a controlled comparison of optimization methods.

In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class DemandShapingStrategy:
    """A demand intervention and the customers it targets."""
    name: str
    strategy_type: str
    parameters: Dict[str, Any]
    effectiveness: float
    cost_factor: float
    target_customers: List[int]


@dataclass
class MarketInfluenceMechanism:
    """A mechanism used to calculate a market-influence score."""
    name: str
    mechanism_type: str
    influence_power: float
    investment_required: float
    expected_roi: float
    time_horizon: int


@dataclass
class StrategicPartnership:
    """A partnership with a customer and its remaining duration."""
    partner_id: int
    partnership_type: str
    commitment_level: float
    mutual_benefits: Dict[str, float]
    contract_duration: int


@dataclass
class MarketPosition:
    """Bounded indicators describing the simulated market position."""
    market_share: float
    competitive_advantage: float
    brand_strength: float
    customer_loyalty: float
    innovation_leadership: float


@dataclass
class StrategicOutcome:
    """The state and reported measures for one simulation period."""
    period: int
    demand_shaped: Dict[int, float]
    market_influence: Dict[str, float]
    partnerships_active: List[StrategicPartnership]
    competitive_position: MarketPosition
    financial_performance: Dict[str, float]
    strategic_value: float


class ProactiveMarketShaper:
    """Simulate proactive demand shaping and market orchestration."""

    def __init__(self, depot, customers, vehicles, num_periods):
        self.depot = depot
        self.customers = customers
        self.vehicles = vehicles
        self.num_periods = num_periods
        self.demand_shaping_strategies = []
        self.market_influence_mechanisms = []
        self.strategic_partnerships = []
        self.market_position = MarketPosition(0.1, 0.2, 0.3, 0.4, 0.2)
        self.strategic_outcomes = []
        self.market_evolution = []
        self.competitive_dynamics = []
        self.strategic_metrics = {
            "demand_shaping_effectiveness": [],
            "market_influence_power": [],
            "partnership_value": [],
            "competitive_advantage": [],
            "strategic_roi": [],
        }
        print(
            f"Initialized Proactive Market Shaper for {len(customers)} "
            f"customers and {len(vehicles)} vehicles."
        )

    def initialize_demand_shaping_strategies(self) -> None:
        """Set up the four demand-shaping rules used in the simulation."""
        print("\n=== INITIALIZING DEMAND-SHAPING STRATEGIES ===")
        self.demand_shaping_strategies = [
            DemandShapingStrategy(
                "Dynamic Pricing", "pricing",
                {"price_elasticity": -0.3, "price_range": 0.2,
                 "adjustment_frequency": 1},
                0.75, 0.05, list(range(1, len(self.customers) + 1)),
            ),
            DemandShapingStrategy(
                "Behavioral Nudging", "nudging",
                {"nudge_strength": 0.4, "nudge_frequency": 2,
                 "nudge_types": ["timing", "quantity", "frequency"]},
                0.65, 0.03, [1, 2, 3, 4, 5],
            ),
            DemandShapingStrategy(
                "Market Segmentation", "segmentation",
                {"segment_count": 3, "targeting_precision": 0.8,
                 "customization_level": 0.6},
                0.70, 0.08, list(range(1, len(self.customers) + 1)),
            ),
            DemandShapingStrategy(
                "Loyalty Program", "loyalty",
                {"loyalty_points_rate": 0.1, "redemption_threshold": 100,
                 "tier_levels": 3},
                0.80, 0.06, [1, 3, 5, 7, 9],
            ),
        ]
        for strategy in self.demand_shaping_strategies:
            strategy.target_customers = [
                customer_id for customer_id in strategy.target_customers
                if customer_id in {customer.id for customer in self.customers}
            ]
        print(f"Initialized {len(self.demand_shaping_strategies)} strategies:")
        for strategy in self.demand_shaping_strategies:
            print(
                f"  - {strategy.name}: {strategy.strategy_type}; "
                f"effectiveness {strategy.effectiveness:.2f}"
            )

    def initialize_market_influence_mechanisms(self) -> None:
        """Set up the market-influence mechanisms and their parameters."""
        print("\n=== INITIALIZING MARKET-INFLUENCE MECHANISMS ===")
        self.market_influence_mechanisms = [
            MarketInfluenceMechanism("Partnership Development", "partnership", 0.85, 50000, 2.5, 3),
            MarketInfluenceMechanism("Capacity Allocation", "allocation", 0.70, 30000, 1.8, 2),
            MarketInfluenceMechanism("Competitive Positioning", "positioning", 0.75, 40000, 2.0, 4),
            MarketInfluenceMechanism("Ecosystem Orchestration", "ecosystem", 0.90, 80000, 3.0, 5),
        ]
        print(f"Initialized {len(self.market_influence_mechanisms)} mechanisms:")
        for mechanism in self.market_influence_mechanisms:
            print(
                f"  - {mechanism.name}: {mechanism.mechanism_type}; "
                f"influence power {mechanism.influence_power:.2f}"
            )

    def shape_demand(self, period: int) -> Dict[int, float]:
        """Multiply the applicable strategy effects for each customer."""
        shaped_demand = {}
        print(f"\n=== DEMAND SHAPING: PERIOD {period + 1} ===")
        for customer in self.customers:
            total_effect = 1.0
            for strategy in self.demand_shaping_strategies:
                if customer.id in strategy.target_customers:
                    total_effect *= self._apply_demand_shaping_strategy(
                        customer, strategy, period
                    )
            shaped_demand[customer.id] = customer.demand_mean * total_effect
            print(
                f"  Customer {customer.id}: baseline "
                f"{customer.demand_mean:.1f}, shaped "
                f"{shaped_demand[customer.id]:.1f} units"
            )
        return shaped_demand

    def _apply_demand_shaping_strategy(
        self, customer, strategy: DemandShapingStrategy, period: int
    ) -> float:
        """Calculate one strategy's effectiveness-adjusted demand multiplier."""
        if strategy.strategy_type == "pricing":
            price_change = np.random.uniform(
                -strategy.parameters["price_range"],
                strategy.parameters["price_range"],
            )
            demand_change = 1 + price_change * strategy.parameters["price_elasticity"]
        elif strategy.strategy_type == "nudging":
            frequency = strategy.parameters["nudge_frequency"]
            if period % frequency == 0:
                strength = strategy.parameters["nudge_strength"]
                demand_change = 1 + np.random.uniform(-strength, strength * 0.5)
            else:
                demand_change = 1.0
        elif strategy.strategy_type == "segmentation":
            precision = strategy.parameters["targeting_precision"]
            customization = strategy.parameters["customization_level"]
            demand_change = 1 + precision * customization * 0.2
        elif strategy.strategy_type == "loyalty":
            demand_change = 1 + strategy.parameters["loyalty_points_rate"] * 0.3
        else:
            demand_change = 1.0

        effectiveness_factor = 1 + (strategy.effectiveness - 1) * 0.5
        adjusted_change = 1 + (demand_change - 1) * effectiveness_factor
        return max(0.5, min(1.5, adjusted_change))

    def develop_partnerships(self, period: int) -> List[StrategicPartnership]:
        """Form partnerships probabilistically and age active contracts."""
        print(f"\n=== PARTNERSHIP DEVELOPMENT: PERIOD {period + 1} ===")
        new_partnerships = []
        active_ids = {p.partner_id for p in self.strategic_partnerships}
        for customer in self.customers:
            if customer.id in active_ids or random.random() >= 0.3:
                continue
            if customer.demand_mean > 25:
                partnership_type, commitment, duration = "strategic", 0.8, 5
            elif customer.demand_mean > 15:
                partnership_type, commitment, duration = "preferred", 0.6, 3
            else:
                partnership_type, commitment, duration = "basic", 0.4, 2
            partnership = StrategicPartnership(
                customer.id, partnership_type, commitment,
                {"cost_reduction": 0.15, "service_improvement": 0.20,
                 "demand_stability": 0.25, "innovation_collaboration": 0.10},
                duration,
            )
            self.strategic_partnerships.append(partnership)
            new_partnerships.append(partnership)
            print(
                f"  New partnership: customer {customer.id}, "
                f"{partnership_type}, commitment {commitment:.2f}"
            )

        for partnership in self.strategic_partnerships:
            partnership.contract_duration -= 1
        expired = [p for p in self.strategic_partnerships if p.contract_duration <= 0]
        for partnership in expired:
            print(f"  Partnership expired: customer {partnership.partner_id}")
        self.strategic_partnerships = [
            p for p in self.strategic_partnerships if p.contract_duration > 0
        ]
        print(f"Active partnerships: {len(self.strategic_partnerships)}")
        return new_partnerships

    def orchestrate_market(
        self, period: int, shaped_demand: Dict[int, float]
    ) -> Dict[str, float]:
        """Calculate mechanism scores and update the simulated market state."""
        market_influence = {
            mechanism.name: self._apply_market_influence_mechanism(
                mechanism, period, shaped_demand
            )
            for mechanism in self.market_influence_mechanisms
        }
        self._update_market_position(market_influence)
        self.competitive_dynamics.append(self._calculate_competitive_dynamics())
        print("\n=== MARKET ORCHESTRATION ===")
        for name, influence in market_influence.items():
            print(f"  {name}: {influence:.3f}")
        print(f"  Market share: {self.market_position.market_share:.2%}")
        print(
            "  Competitive advantage: "
            f"{self.market_position.competitive_advantage:.2%}"
        )
        return market_influence

    def _apply_market_influence_mechanism(
        self, mechanism: MarketInfluenceMechanism, period: int,
        shaped_demand: Dict[int, float]
    ) -> float:
        """Evaluate a mechanism using the model's stated influence rules."""
        if mechanism.mechanism_type == "partnership":
            influence = mechanism.influence_power * len(self.strategic_partnerships) * 0.1
        elif mechanism.mechanism_type == "allocation":
            capacity = sum(vehicle.capacity for vehicle in self.vehicles)
            utilization = sum(shaped_demand.values()) / capacity if capacity else 0.0
            influence = mechanism.influence_power * min(1.0, utilization)
        elif mechanism.mechanism_type == "positioning":
            strength = (
                self.market_position.brand_strength
                + self.market_position.innovation_leadership
            )
            influence = mechanism.influence_power * strength * 0.5
        elif mechanism.mechanism_type == "ecosystem":
            strength = len(self.strategic_partnerships) * 0.05
            strength += self.market_position.market_share
            influence = mechanism.influence_power * strength
        else:
            influence = 0.5
        time_factor = 1 - (period % mechanism.time_horizon) / mechanism.time_horizon * 0.3
        return max(0.1, min(1.0, influence * time_factor))

    def _update_market_position(self, market_influence: Dict[str, float]) -> None:
        """Advance market-position indicators using mean mechanism influence."""
        overall = float(np.mean(list(market_influence.values())))
        position = self.market_position
        position.market_share = min(0.5, position.market_share + overall * 0.02)
        position.competitive_advantage = min(
            0.8, position.competitive_advantage + overall * 0.03
        )
        position.brand_strength = min(0.7, position.brand_strength + overall * 0.025)
        position.customer_loyalty = min(
            0.9, position.customer_loyalty + overall * 0.02
        )
        position.innovation_leadership = min(
            0.6, position.innovation_leadership + overall * 0.015
        )

    def _calculate_competitive_dynamics(self) -> Dict[str, float]:
        """Generate illustrative random indicators of competitive conditions."""
        pressure = 0.3 + random.random() * 0.2
        volatility = 0.1 + random.random() * 0.1
        innovation_race = 0.2 + random.random() * 0.3
        return {
            "competitive_pressure": pressure,
            "market_volatility": volatility,
            "innovation_race": innovation_race,
            "market_stability": 1 - volatility,
        }

    def optimize_with_market_shaping(self, period: int) -> StrategicOutcome:
        """Run the model's ordered shaping, partnership, and reporting stages."""
        shaped_demand = self.shape_demand(period)
        self.develop_partnerships(period)
        market_influence = self.orchestrate_market(period, shaped_demand)
        routes = self._optimize_routes_with_shaped_demand(shaped_demand)
        financial = self._calculate_financial_performance(shaped_demand, routes, period)
        strategic_value = self._calculate_strategic_value(market_influence, financial)
        position = MarketPosition(**vars(self.market_position))
        outcome = StrategicOutcome(
            period, shaped_demand, market_influence,
            [StrategicPartnership(**vars(p)) for p in self.strategic_partnerships],
            position, financial, strategic_value,
        )
        self.strategic_outcomes.append(outcome)
        self._update_strategic_metrics(outcome)
        self._report_strategic_results(outcome)
        return outcome

    def _optimize_routes_with_shaped_demand(
        self, shaped_demand: Dict[int, float]
    ) -> Dict[int, List[int]]:
        """Assign demand-sorted customers in consecutive groups to vehicles."""
        if not self.vehicles:
            raise ValueError("At least one vehicle is required for route assignment.")
        customer_demands = sorted(
            ((customer.id, shaped_demand[customer.id]) for customer in self.customers),
            key=lambda item: item[1], reverse=True,
        )
        per_vehicle = len(self.customers) // len(self.vehicles)
        routes = {}
        for index, vehicle in enumerate(self.vehicles):
            start = index * per_vehicle
            end = start + per_vehicle
            if index == len(self.vehicles) - 1:
                end = len(customer_demands)
            routes[vehicle.id] = [
                customer_id for customer_id, _ in customer_demands[start:end]
            ]
        return routes

    def _calculate_financial_performance(
        self, shaped_demand: Dict[int, float],
        routes: Dict[int, List[int]], period: int
    ) -> Dict[str, float]:
        """Estimate revenue, simplified costs, profit, and profit margin."""
        customer_by_id = {customer.id: customer for customer in self.customers}
        vehicle_by_id = {vehicle.id: vehicle for vehicle in self.vehicles}
        partner_ids = {p.partner_id for p in self.strategic_partnerships}
        revenue = sum(
            demand * 10 * (1.1 if customer_id in partner_ids else 1.0)
            for customer_id, demand in shaped_demand.items()
        )
        transport_cost = 0.0
        for vehicle_id, customer_ids in routes.items():
            if customer_ids:
                vehicle = vehicle_by_id[vehicle_id]
                transport_cost += vehicle.fixed_cost
                for customer_id in customer_ids:
                    customer = customer_by_id[customer_id]
                    distance = np.hypot(
                        customer.x - self.depot.x, customer.y - self.depot.y
                    )
                    transport_cost += distance * vehicle.variable_cost
        strategy_costs = sum(
            strategy.cost_factor * revenue
            for strategy in self.demand_shaping_strategies
        )
        partnership_costs = len(self.strategic_partnerships) * 1000
        total_cost = transport_cost + strategy_costs + partnership_costs
        profit = revenue - total_cost
        margin = profit / revenue if revenue > 0 else 0.0
        return {
            "revenue": revenue,
            "cost": total_cost,
            "profit": profit,
            "profit_margin": margin,
            "strategy_costs": strategy_costs,
            "partnership_costs": partnership_costs,
            "transport_costs": transport_cost,
        }

    def _calculate_strategic_value(
        self, market_influence: Dict[str, float],
        financial_performance: Dict[str, float]
    ) -> float:
        """Combine the four weighted indicators defined for this simulation."""
        financial_value = financial_performance["profit_margin"] * 0.4
        market_value = float(np.mean(list(market_influence.values()))) * 0.3
        partnership_value = len(self.strategic_partnerships) * 0.02 * 0.2
        competitive_value = self.market_position.competitive_advantage * 0.1
        return financial_value + market_value + partnership_value + competitive_value

    def _update_strategic_metrics(self, outcome: StrategicOutcome) -> None:
        """Record period indicators so trends can be summarized later."""
        baseline = sum(customer.demand_mean for customer in self.customers)
        shaped = sum(outcome.demand_shaped.values())
        effectiveness = shaped / baseline if baseline else 1.0
        metrics = self.strategic_metrics
        metrics["demand_shaping_effectiveness"].append(effectiveness)
        metrics["market_influence_power"].append(
            float(np.mean(list(outcome.market_influence.values())))
        )
        metrics["partnership_value"].append(len(outcome.partnerships_active) * 0.1)
        metrics["competitive_advantage"].append(
            outcome.competitive_position.competitive_advantage
        )
        denominator = outcome.financial_performance["strategy_costs"] + 1
        metrics["strategic_roi"].append(
            outcome.financial_performance["profit"] / denominator
        )

    def _report_strategic_results(self, outcome: StrategicOutcome) -> None:
        """Print the principal indicators for one period."""
        financial = outcome.financial_performance
        print(f"\n=== PERIOD {outcome.period + 1} RESULTS ===")
        print(f"  Revenue: ${financial['revenue']:.2f}")
        print(f"  Cost: ${financial['cost']:.2f}")
        print(f"  Profit: ${financial['profit']:.2f}")
        print(f"  Profit margin: {financial['profit_margin']:.2%}")
        print(f"  Strategic value: {outcome.strategic_value:.3f}")
        print(f"  Active partnerships: {len(outcome.partnerships_active)}")

    def run_strategic_orchestration(self) -> Dict[str, Any]:
        """Initialize components, run every period, and aggregate outcomes."""
        self.initialize_demand_shaping_strategies()
        self.initialize_market_influence_mechanisms()
        for period in range(self.num_periods):
            print(f"\n=== STRATEGIC ORCHESTRATION: PERIOD {period + 1} ===")
            self.optimize_with_market_shaping(period)
        final_results = self._calculate_final_results()
        print("\n=== SIMULATION SUMMARY ===")
        print(f"  Total profit: ${final_results['total_profit']:.2f}")
        print(f"  Average strategic value: {final_results['avg_strategic_value']:.3f}")
        print(f"  Final market share: {final_results['final_market_share']:.2%}")
        print(f"  Active partnerships at horizon end: {final_results['total_partnerships']}")
        return final_results

    def _calculate_final_results(self) -> Dict[str, Any]:
        """Aggregate period outcomes and metric trends."""
        if not self.strategic_outcomes:
            return {}
        outcomes = self.strategic_outcomes
        total_profit = sum(o.financial_performance["profit"] for o in outcomes)
        total_revenue = sum(o.financial_performance["revenue"] for o in outcomes)
        avg_margin = float(np.mean([
            o.financial_performance["profit_margin"] for o in outcomes
        ]))
        avg_value = float(np.mean([o.strategic_value for o in outcomes]))
        partnership_types = [p.partnership_type for p in self.strategic_partnerships]
        distribution = {
            kind: partnership_types.count(kind) for kind in sorted(set(partnership_types))
        }
        trends = {}
        for name, values in self.strategic_metrics.items():
            if values:
                trends[name] = {
                    "initial": values[0], "final": values[-1],
                    "improvement": values[-1] - values[0],
                    "average": float(np.mean(values)),
                }
        evolution = {
            "market_share_progression": [o.competitive_position.market_share for o in outcomes],
            "competitive_advantage_progression": [
                o.competitive_position.competitive_advantage for o in outcomes
            ],
            "strategic_value_progression": [o.strategic_value for o in outcomes],
        }
        return {
            "total_profit": total_profit,
            "total_revenue": total_revenue,
            "avg_profit_margin": avg_margin,
            "avg_strategic_value": avg_value,
            "final_market_share": outcomes[-1].competitive_position.market_share,
            "final_competitive_advantage": outcomes[-1].competitive_position.competitive_advantage,
            "total_partnerships": len(self.strategic_partnerships),
            "partnership_distribution": distribution,
            "market_evolution": evolution,
            "performance_trends": trends,
            "strategic_outcomes": outcomes,
            "strategic_metrics": self.strategic_metrics,
        }

    def analyze_competitive_landscape(self) -> Dict[str, Any]:
        """Contrast simulated position with illustrative competitor references."""
        competitors = [
            {"name": "Competitor A", "market_share": 0.25, "strength": "Cost Leadership"},
            {"name": "Competitor B", "market_share": 0.20, "strength": "Innovation"},
            {"name": "Competitor C", "market_share": 0.15, "strength": "Service Quality"},
            {"name": "Competitor D", "market_share": 0.10, "strength": "Niche Focus"},
        ]
        positioning = {
            "our_position": {
                "market_share": self.market_position.market_share,
                "strength": "Strategic Orchestration",
                "competitive_advantage": self.market_position.competitive_advantage,
            },
            "competitors": competitors,
        }
        recommendations = []
        if self.market_position.market_share < 0.15:
            recommendations.append("Assess partnership options for market development.")
        if self.market_position.competitive_advantage < 0.5:
            recommendations.append("Assess investments in innovation and differentiation.")
        if len(self.strategic_partnerships) < 5:
            recommendations.append("Review the scale and value of the partnership network.")
        print("\n=== COMPETITIVE LANDSCAPE ===")
        print(f"Our simulated market share: {self.market_position.market_share:.2%}")
        print("Illustrative competitor reference shares:")
        for competitor in competitors:
            print(f"  {competitor['name']}: {competitor['market_share']:.2%}")
        for index, recommendation in enumerate(recommendations, 1):
            print(f"  {index}. {recommendation}")
        return {
            "market_positioning": positioning,
            "recommendations": recommendations,
            "competitive_threats": ["Price competition", "Innovation", "Market consolidation"],
            "growth_opportunities": ["Emerging markets", "Service expansion", "Technology"],
        }

In [ ]:
def create_market_shaping_instance():
    """Create the depot, ten generated customers, and three vehicles."""
    depot = type("Depot", (), {
        "x": 0, "y": 0, "initial_inventory": 1200, "holding_cost": 0.1
    })()

    np.random.seed(42)
    random.seed(42)
    customers = []
    for index in range(10):
        center_x = np.random.choice([-12, 0, 12])
        center_y = np.random.choice([-12, 0, 12])
        x = center_x + np.random.normal(0, 5)
        y = center_y + np.random.normal(0, 5)
        customers.append(type("Customer", (), {
            "id": index + 1,
            "x": x,
            "y": y,
            "demand_mean": np.random.uniform(18, 32),
            "demand_std": np.random.uniform(4, 7),
            "holding_cost": np.random.uniform(0.1, 0.3),
            "initial_inventory": np.random.uniform(45, 85),
            "min_inventory": np.random.uniform(18, 28),
            "max_inventory": np.random.uniform(95, 135),
        })())

    vehicles = [
        type("Vehicle", (), {
            "id": vehicle_id, "capacity": 140,
            "fixed_cost": 200, "variable_cost": 2.8,
        })()
        for vehicle_id in range(1, 4)
    ]
    return depot, customers, vehicles


print("Creating the illustrative market-shaping instance...")
depot, customers, vehicles = create_market_shaping_instance()
print(f"Depot location: ({depot.x}, {depot.y})")
print(f"Customers: {len(customers)}")
for customer in customers:
    print(
        f"  C{customer.id}: location ({customer.x:.1f}, {customer.y:.1f}), "
        f"mean demand {customer.demand_mean:.1f} units"
    )
print(f"Vehicles: {len(vehicles)}; capacity parameter: {vehicles[0].capacity} units each")

In [ ]:
def run_market_shaping_optimization(depot, customers, vehicles, num_periods=5):
    """Run the fixed-sequence market-shaping simulation."""
    market_shaper = ProactiveMarketShaper(
        depot, customers, vehicles, num_periods
    )
    print(
        f"\nSimulation size: {len(customers)} customers, "
        f"{len(vehicles)} vehicles, {num_periods} periods."
    )
    start_time = time.time()
    results = market_shaper.run_strategic_orchestration()
    elapsed = time.time() - start_time
    print(f"Simulation runtime: {elapsed:.2f} seconds")
    return market_shaper, results


market_shaper, results = run_market_shaping_optimization(
    depot, customers, vehicles, num_periods=5
)

In [ ]:
def analyze_market_shaping_performance(market_shaper, results):
    """Summarize financial, market, partnership, and trend indicators."""
    print("\n=== MARKET-SHAPING PERFORMANCE ANALYSIS ===")
    print(f"Total estimated revenue: ${results['total_revenue']:.2f}")
    print(f"Total estimated profit: ${results['total_profit']:.2f}")
    print(f"Mean period profit margin: {results['avg_profit_margin']:.2%}")
    print(f"Mean strategic-value score: {results['avg_strategic_value']:.3f}")
    print(f"Final simulated market share: {results['final_market_share']:.2%}")
    print(
        "Change from the initialized market-share indicator: "
        f"{results['final_market_share'] - 0.1:+.2%}"
    )
    print(f"Active partnerships at horizon end: {results['total_partnerships']}")
    print(f"Partnership types at horizon end: {results['partnership_distribution']}")

    print("\nPeriod metric trends:")
    for name, trend in results["performance_trends"].items():
        print(
            f"  {name.replace('_', ' ').title()}: "
            f"initial {trend['initial']:.3f}, final {trend['final']:.3f}, "
            f"change {trend['improvement']:+.3f}, mean {trend['average']:.3f}"
        )

    evolution = results["market_evolution"]
    first_share = evolution["market_share_progression"][0]
    last_share = evolution["market_share_progression"][-1]
    absolute_change = last_share - first_share
    relative_change = absolute_change / first_share if first_share else float("nan")
    print(
        "\nMarket-share change across recorded periods: "
        f"{absolute_change:+.2%} absolute; {relative_change:+.1%} relative."
    )
    competitive = market_shaper.analyze_competitive_landscape()
    return {
        "financial_performance": {
            "total_revenue": results["total_revenue"],
            "total_profit": results["total_profit"],
            "mean_period_profit_margin": results["avg_profit_margin"],
        },
        "market_position": {
            "market_share": results["final_market_share"],
            "competitive_advantage": results["final_competitive_advantage"],
            "market_share_change": absolute_change,
        },
        "partnership_success": {
            "active_at_horizon_end": results["total_partnerships"],
            "partnership_distribution": results["partnership_distribution"],
        },
        "strategy_effectiveness": {
            name: trend["average"]
            for name, trend in results["performance_trends"].items()
        },
        "competitive_analysis": competitive,
    }


market_shaping_analysis = analyze_market_shaping_performance(
    market_shaper, results
)

In [ ]:
def visualize_market_shaping_results(market_shaper, results, analysis):
    """Plot simulated period outcomes and the illustrative reference landscape."""
    outcomes = results["strategic_outcomes"]
    periods = np.arange(1, len(outcomes) + 1)
    fig, axes = plt.subplots(3, 3, figsize=(18, 15))
    fig.suptitle("Proactive Demand and Market-Shaping Simulation", fontsize=16)

    ax = axes[0, 0]
    profits = [o.financial_performance["profit"] for o in outcomes]
    revenues = [o.financial_performance["revenue"] for o in outcomes]
    ax.plot(periods, profits, "o-", label="Profit")
    ax.plot(periods, revenues, "s-", label="Revenue")
    ax.set(title="Estimated Financial Performance", xlabel="Period", ylabel="Amount ($)")
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[0, 1]
    evolution = results["market_evolution"]
    ax.plot(periods, evolution["market_share_progression"], "o-", label="Market share")
    ax.plot(periods, evolution["competitive_advantage_progression"], "s-", label="Competitive advantage")
    ax.plot(periods, evolution["strategic_value_progression"], "^-", label="Strategic-value score")
    ax.set(title="Simulated State Indicators", xlabel="Period", ylabel="Indicator value")
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[0, 2]
    distribution = results["partnership_distribution"]
    if distribution:
        ax.bar(list(distribution), list(distribution.values()))
    else:
        ax.text(0.5, 0.5, "No active partnerships", ha="center", va="center", transform=ax.transAxes)
    ax.set(title="Active Partnership Types at Horizon End", ylabel="Number of partnerships")
    ax.tick_params(axis="x", rotation=35)
    ax.grid(True, axis="y", alpha=0.3)

    ax = axes[1, 0]
    metrics = results["strategic_metrics"]
    for name in ("demand_shaping_effectiveness", "market_influence_power", "partnership_value"):
        ax.plot(periods, metrics[name], marker="o", label=name.replace("_", " ").title())
    ax.set(title="Selected Period Metrics", xlabel="Period", ylabel="Metric value")
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[1, 1]
    original = [customer.demand_mean for customer in market_shaper.customers]
    final_demand = outcomes[-1].demand_shaped
    shaped = [final_demand[customer.id] for customer in market_shaper.customers]
    x = np.arange(len(original))
    width = 0.38
    ax.bar(x - width / 2, original, width, label="Baseline mean demand")
    ax.bar(x + width / 2, shaped, width, label="Shaped demand")
    ax.set(title="Baseline and Final-Period Demand", xlabel="Customer ID", ylabel="Demand (units)")
    ax.set_xticks(x, [customer.id for customer in market_shaper.customers])
    ax.legend()
    ax.grid(True, axis="y", alpha=0.3)

    ax = axes[1, 2]
    landscape = analysis["competitive_analysis"]["market_positioning"]
    names = ["Our simulated position"] + [c["name"] for c in landscape["competitors"]]
    shares = [landscape["our_position"]["market_share"]] + [
        c["market_share"] for c in landscape["competitors"]
    ]
    ax.bar(names, shares)
    ax.set(title="Market-Share Indicators", ylabel="Market share (fraction)")
    ax.tick_params(axis="x", rotation=35)
    ax.grid(True, axis="y", alpha=0.3)

    ax = axes[2, 0]
    ax.plot(periods, metrics["strategic_roi"], "o-", color="tab:purple", label="Profit / (strategy cost + 1)")
    ax.set(title="Defined Strategic-ROI Proxy", xlabel="Period", ylabel="Ratio")
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[2, 1]
    average_margin = float(np.mean([
        o.financial_performance["profit_margin"] for o in outcomes
    ]))
    average_influence = float(np.mean([
        np.mean(list(o.market_influence.values())) for o in outcomes
    ]))
    final_partnership_value = len(outcomes[-1].partnerships_active) * 0.02
    final_advantage = outcomes[-1].competitive_position.competitive_advantage
    components = ["Financial (0.4 × margin)", "Influence (0.3 × mean)",
                  "Partnership (0.2 × term)", "Advantage (0.1 × term)"]
    values = [0.4 * average_margin, 0.3 * average_influence,
              0.2 * final_partnership_value, 0.1 * final_advantage]
    ax.bar(components, values)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set(title="Illustrative Strategic-Value Components", ylabel="Weighted contribution")
    ax.tick_params(axis="x", rotation=25)
    ax.grid(True, axis="y", alpha=0.3)

    ax = axes[2, 2]
    ax.plot(periods, metrics["strategic_roi"], "o-", color="tab:green", label="Strategic-ROI proxy")
    ax.set_xlabel("Period")
    ax.set_ylabel("Strategic-ROI proxy", color="tab:green")
    ax.tick_params(axis="y", labelcolor="tab:green")
    ax.grid(True, alpha=0.3)
    ax2 = ax.twinx()
    ax2.plot(periods, evolution["market_share_progression"], "s-", color="tab:blue", label="Market share")
    ax2.set_ylabel("Market share (fraction)", color="tab:blue")
    ax2.tick_params(axis="y", labelcolor="tab:blue")
    ax.set_title("ROI Proxy and Market Share")
    ax.legend(loc="upper left")
    ax2.legend(loc="upper right")

    fig.tight_layout()
    plt.show()


visualize_market_shaping_results(market_shaper, results, market_shaping_analysis)

In [ ]:
def compare_market_shaping_with_approaches(market_shaper, results, analysis):
    """Contrast simulation indicators with explicitly illustrative reference values."""
    approaches = [
        "Reactive Optimization", "Operational Efficiency",
        "Customer Service", "Strategic Orchestration",
    ]
    # The first three rows are illustrative reference assumptions, not measured results.
    influence = {
        "Reactive Optimization": 0.10,
        "Operational Efficiency": 0.25,
        "Customer Service": 0.45,
        "Strategic Orchestration": analysis["strategy_effectiveness"]["market_influence_power"],
    }
    advantage = {
        "Reactive Optimization": 0.15,
        "Operational Efficiency": 0.35,
        "Customer Service": 0.55,
        "Strategic Orchestration": results["final_competitive_advantage"],
    }
    value = {
        "Reactive Optimization": 0.20,
        "Operational Efficiency": 0.40,
        "Customer Service": 0.65,
        "Strategic Orchestration": results["avg_strategic_value"],
    }
    growth = {
        "Reactive Optimization": 0.05,
        "Operational Efficiency": 0.15,
        "Customer Service": 0.25,
        "Strategic Orchestration": results["final_market_share"] - 0.1,
    }
    print("\n=== ILLUSTRATIVE APPROACH COMPARISON ===")
    print("Reference values are assumptions; only the orchestration row uses this run's indicators.")
    print(f"{'Approach':<26}{'Influence':>12}{'Advantage':>12}{'Value':>10}{'Growth':>10}")
    for approach in approaches:
        print(
            f"{approach:<26}{influence[approach]:>12.2f}"
            f"{advantage[approach]:>12.2f}{value[approach]:>10.2f}"
            f"{growth[approach]:>10.2f}"
        )

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    panels = [
        (influence, "Market-influence indicator", axes[0, 0]),
        (advantage, "Competitive-advantage indicator", axes[0, 1]),
        (value, "Strategic-value indicator", axes[1, 0]),
        (growth, "Market-share change from 10% baseline", axes[1, 1]),
    ]
    for values, title, ax in panels:
        ax.bar(approaches, [values[name] for name in approaches])
        ax.set_title(title)
        ax.tick_params(axis="x", rotation=30)
        ax.axhline(0, color="black", linewidth=0.8)
        ax.grid(True, axis="y", alpha=0.3)
    fig.tight_layout()
    plt.show()

    return {
        "reference_values_are_illustrative": True,
        "strategic_advantage_above_0_5": results["final_competitive_advantage"] > 0.5,
        "market_share_above_0_2": results["final_market_share"] > 0.2,
        "positive_mean_strategic_value": results["avg_strategic_value"] > 0,
        "positive_strategy_roi_proxy": analysis["strategy_effectiveness"]["strategic_roi"] > 0,
    }


comparison_results = compare_market_shaping_with_approaches(
    market_shaper, results, market_shaping_analysis
)

## Interpreting the simulation

### Demand-shaping strategies

Dynamic pricing, behavioral nudging, segmentation, and loyalty programs alter the demand inputs through the multiplier rules in the code. The price elasticity is configured as $-0.3$, so a positive price change reduces the unadjusted demand multiplier. The multiplier also depends on a random price change and the strategy's effectiveness factor. A measured shaped-demand total above the baseline is not, by itself, evidence of greater customer welfare or greater profit: the model does not estimate consumer surplus, price-dependent revenue, or service consequences of demand changes.

### Partnerships and market influence

Partnership formation is probabilistic, with customer type selected from baseline mean demand. The partnership list is aged after formation in each period; therefore, a newly formed contract has its remaining duration reduced during that same period. Mechanism scores are constructed indicators based on partnership count, demand relative to aggregate vehicle capacity, and the current market-position state. Mechanism investment and expected-ROI fields are stored as parameters but are not included in the financial calculation.

The market-position indicators are updated by bounded increments. They represent the simulation's state variables, not estimates inferred from real market data. The competitor shares in the landscape display are reference assumptions and are not generated by the customer or market model.

### Financial and strategic indicators

Period revenue uses shaped demand and a fixed unit price, with a premium multiplier for active partners. Estimated cost includes strategy charges, partnership charges, vehicle fixed costs, and depot-to-customer distance charges. The margin can be negative when estimated cost exceeds revenue. The strategic-value score deliberately combines a financial term with influence, partnership, and competitive-position terms; because these terms have different interpretations and scales, the score should be read as a model-defined index rather than a direct monetary value.

The `strategic_roi` metric is specifically defined as profit divided by strategy costs plus one. It is a proxy, not a conventional return-on-investment calculation: it excludes investment requirements and partnership costs from its denominator. A negative value indicates negative profit under this proxy's numerator.

### Relationship to the Inventory-Routing Problem

An IRP ordinarily connects inventory decisions, delivery quantities, vehicle capacities, and route costs across a planning horizon. This notebook retains its stated market-shaping simulation sequence but models only a simplified demand-sorted customer-to-vehicle assignment and estimated costs. Depot and customer inventory attributes are generated but are not updated or used to decide deliveries. The resulting routes are therefore assignments, not validated feasible vehicle tours, and the simulation does not establish an optimal IRP solution.

### Limitations and natural extensions

The model omits inventory balance, replenishment quantities, time windows, vehicle-load feasibility, return travel, inter-customer travel, demand uncertainty around `demand_mean`, and calibrated market-response data. These omissions limit operational interpretation. Adding such details would require explicitly extending the model and its constraints; they are not represented by the current simulation. Likewise, the comparison chart is illustrative rather than a controlled experiment, so it cannot support a causal claim that one approach outperforms another.